In [94]:
import requests
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

donnees = []
scraper = cloudscraper.create_scraper()

for j in range(1,30):
    if j==1:  
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld69#list")
    else:
        page = requests.get("https://www.etreproprio.com/annonces/thf.ld69.odd.g"+str(j)+"#list")
        
    soup = BeautifulSoup(page.content, "html.parser")
    temp = soup.find("div", class_='ep-search-list-wrapper').find_all("a")
    
    liens = []
    for i in range(len(temp)):
        if "https://www.etreproprio.com/immobilier-" in temp[i]['href']:
            liens.append(temp[i]['href'])
    
    for lien in liens:
        try:
            page_annonce = scraper.get(lien)
            soup_ = BeautifulSoup(page_annonce.content, 'lxml')

            try:
                # "Vente Appartement 75 m² Paris-14e"
                titre = soup_.find("title").text.strip()
                titre = unicodedata.normalize("NFKC", titre)
            except:
                titre = None

            try:
                # "Appartement 75 m² à Paris-14e"
                description_h1 = soup_.find("h1", class_='annonce-immobilier').text.strip()
                description_h1 = unicodedata.normalize("NFKC", description_h1)
            except:
                description_h1 = None

            try:
                # Extrait la ville depuis le titre : "Vente Appartement 75 m² Paris-14e" → "Paris-14e"
                ville = soup_.find("title").text.strip().split("²")[-1].strip()
            except:
                ville = None

            try:
                prix = soup_.find("div", class_='ep-price').text.strip()
                prix = int(prix.replace(" ", "").replace("\xa0", "").replace("€", ""))
            except:
                prix = None

            try:
                surface = soup_.find("div", class_='ep-area').text.strip()
                surface = int(surface.replace("\xa0", "").replace("m²", "").strip())
            except:
                surface = None

            try:
                type_bien = titre.split(" ")[1]  # le 2ème mot après "Vente"
            except:
                type_bien = None

            try:
                desc_texte = soup_.find("div", class_='ep-desc-truncated').text.strip()
                desc_texte = unicodedata.normalize("NFKC", desc_texte)
            except:
                desc_texte = None
            
            try:
                etage = None
                if desc_texte:
                # Cas 1 — chiffres
                    etage_match = re.search(r'(\d+)\s*(?:[eèéê?][mr]?[eèé]?|°)?\s*(?:et\s+)?(?:\(dernier\)|dernier)?\s*[eé]tage', desc_texte, re.IGNORECASE)
                    if etage_match:
                        etage = int(etage_match.group(1))
                # Cas 2 — lettres
                    else:
                        etages_lettres = {
                            "premier": 1, "première": 1, "premiere": 1,
                            "deuxième": 2, "deuxieme": 2,
                            "troisième": 3, "troisieme": 3,
                            "quatrième": 4, "quatrieme": 4,
                            "cinquième": 5, "cinquieme": 5,
                            "sixième": 6, "sixieme": 6,
                            "septième": 7, "septieme": 7,
                            "huitième": 8, "huitieme": 8,
                            "neuvième": 9, "neuvieme": 9,
                            "dixième": 10, "dixieme": 10,
                        }
                        for mot, num in etages_lettres.items():
                            if mot in desc_texte.lower():
                                etage = num
                                break
                # Cas 3 — rez-de-chaussée
                    if etage is None:
                        rdc_patterns = ["rez-de-chaussée", "rez de chaussée","rez-de-chaussee", "rez de chaussee", "rdc", "r.d.c"]
                        if any(p in desc_texte.lower() for p in rdc_patterns):
                            etage = 0
            except:
                etage = None

            try:
                dpe = soup_.find("div", class_='dpe-diagram').find("div", class_='selected').text.strip()
            except:
                dpe = None

            try:
                pieces = soup_.find("div", class_='ep-room').text.strip()
                pieces = int(pieces.replace("pièces", "").replace("pièce", "").strip())
            except:
                pieces = None

            print(f"Titre: {titre} | Ville: {ville} | Prix: {prix} | Surface: {surface} | Type: {type_bien} | Etage: {etage} | Pièces: {pieces} | DPE: {dpe}")

            donnees.append({
                "titre": titre,
                "description": description_h1,
                "ville": ville,
                "type_bien": type_bien,
                "etage": etage,
                "prix": prix,
                "surface": surface,
                "pieces": pieces,
                "DPE": dpe,
                "full_description": desc_texte,
                "lien": lien
            })

        except Exception as e:
            print(f"Erreur sur {lien} : {e}")
            continue

# Création du DataFrame
df = pd.DataFrame(donnees)
df["prix"] = pd.to_numeric(df["prix"], errors="coerce")
df["surface"] = pd.to_numeric(df["surface"], errors="coerce")
df["pieces"] = pd.to_numeric(df["pieces"], errors="coerce").astype("Int64")
df["etage"] = pd.to_numeric(df["etage"], errors="coerce").astype("Int64")

print(df)

Titre: Vente Appartement 72 m2 Francheville | Ville: Francheville | Prix: 269000 | Surface: 72 | Type: Appartement | Etage: None | Pièces: 3 | DPE: D
Titre: Vente Appartement 78 m2 Villeurbanne | Ville: Villeurbanne | Prix: 293000 | Surface: 78 | Type: Appartement | Etage: 3 | Pièces: 4 | DPE: C
Titre: Vente Appartement 208 m2 Feyzin | Ville: Feyzin | Prix: 380000 | Surface: 208 | Type: Appartement | Etage: 0 | Pièces: 5 | DPE: A
Titre: Vente Appartement 106 m2 Villefranche-Sur-Saone | Ville: Villefranche-Sur-Saone | Prix: 239000 | Surface: 106 | Type: Appartement | Etage: None | Pièces: 5 | DPE: D
Titre: Vente Maison 154 m2 Mions | Ville: Mions | Prix: 719000 | Surface: None | Type: Maison | Etage: None | Pièces: 5 | DPE: B
Titre: Vente Appartement 152 m2 Lyon-6e | Ville: Lyon-6e | Prix: 645000 | Surface: 152 | Type: Appartement | Etage: 1 | Pièces: 5 | DPE: E
Titre: Vente Appartement 80 m2 Rillieux-La-Pape | Ville: Rillieux-La-Pape | Prix: 299000 | Surface: None | Type: Appartement |

In [95]:
df

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
0,Vente Appartement 72 m2 Francheville,Appartement 72 m2 à Francheville,Francheville,Appartement,<NA>,269000,72.0,3,D,Decultieux Immobililer la référence depuis 196...,https://www.etreproprio.com/immobilier-2529512...
1,Vente Appartement 78 m2 Villeurbanne,Appartement 78 m2 à Villeurbanne,Villeurbanne,Appartement,3,293000,78.0,4,C,Une exclusivité DECULTIEUX Immobilier\nVILLEUR...,https://www.etreproprio.com/immobilier-2529512...
2,Vente Appartement 208 m2 Feyzin,EXCLUSIVITÉ Feyzin La Bégude Appartement r...,Feyzin,Appartement,0,380000,208.0,5,A,FEYZIN LA BÉGUDE Le confort d’une maison réc...,https://www.etreproprio.com/immobilier-2529551...
3,Vente Appartement 106 m2 Villefranche-Sur-Saone,6732- VILLEFRANCHE S/S Centre ville,Villefranche-Sur-Saone,Appartement,<NA>,239000,106.0,5,D,None,https://www.etreproprio.com/immobilier-2529521...
4,Vente Maison 154 m2 Mions,Maison d’architecte 2022 avec piscine et prest...,Mions,Maison,<NA>,719000,NaN,5,B,"Située au fond d’une impasse, dans un secteur ...",https://www.etreproprio.com/immobilier-2529560...
...,...,...,...,...,...,...,...,...,...,...,...
575,Vente Appartement 64 m2 Lyon-9e,Appartement lumineux avec magnifique vue proch...,Lyon-9e,Appartement,9,285000,64.0,3,C,L'Agence SuitHome vous propose en exclusivité ...,https://www.etreproprio.com/immobilier-2526689...
576,Vente Appartement 75 m2 Lyon-9e,"T3 de 73,6 m2 à Lyon 9 au prix de vente de 182...",Lyon-9e,Appartement,2,182300,75.0,3,C,Découvrez en exclusivité chez Lyon Métropole H...,https://www.etreproprio.com/immobilier-2526819...
577,Vente Appartement 84 m2 Tarare,Appartement 84 m2 à Tarare,Tarare,Appartement,0,120000,84.0,4,D,"A venir découvrir rapidement, à TARARE Centre ...",https://www.etreproprio.com/immobilier-2527082...
578,Vente Appartement 77 m2 Lyon-9e,"Appartement T4 de 78 m2 à Lyon 69009, Rue Féli...",Lyon-9e,Appartement,<NA>,375000,77.0,4,C,"Pour toute question ou demande d'information, ...",https://www.etreproprio.com/immobilier-2526785...


In [96]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               197
prix                  0
surface             141
pieces                7
DPE                  58
full_description      9
lien                  0
dtype: int64

In [97]:
extracted = df["titre"].str.extract(r'(\d+(?:[.,]\d+)?)\s*m\S*')[0].str.replace(",", ".", regex=False).astype(float)
extracted

0       72.0
1       78.0
2      208.0
3      106.0
4      154.0
       ...  
575     64.0
576     75.0
577     84.0
578     77.0
579    103.0
Name: 0, Length: 580, dtype: float64

In [98]:
extracted

0       72.0
1       78.0
2      208.0
3      106.0
4      154.0
       ...  
575     64.0
576     75.0
577     84.0
578     77.0
579    103.0
Name: 0, Length: 580, dtype: float64

In [99]:
df["surface"] = df["surface"].fillna(extracted)

In [100]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               197
prix                  0
surface               3
pieces                7
DPE                  58
full_description      9
lien                  0
dtype: int64

In [101]:
print(df.loc[df["surface"].isna(), "titre"].iloc[0])
print(df.loc[df["surface"].isna(), "description"].iloc[0])
print(df.loc[df["surface"].isna(), "full_description"].iloc[0])

Vente Appartement  Villeurbanne
Appartement   m2
IN'LI AURA vend à VILLEURBANNE (69100), un appartement T2 de 54.60 m2 au 8ème et dernier étage. Réf. 202985. Lot n°24

Dans copropriété de 72 logements construite en 1994 situé au 57/59 rue Descartes à VILLEURBANNE, proche métro République, à côté du Parc Chanteur.

Au 8eme et dernier étage avec ascenseur, agréable T2 de 54,6 m2 traversant expo EST/OUEST, clair et lumineux.

Ce bien est composé d'un séjour de 19m2 ouvrant sur une terrasse de 10m2, la cuisine de 10,7 m2 peut être ouverte sur le séjour, une chambre avec vue sur Fourvière de 12,2 m2 avec dressing attenant, une salle de bains et un WC indépendant.
Chauffage individuel gaz, double vitrage PVC.

Atouts supplémentaires :
-             Terrasse de 10m2
-             Parquet en chêne dans les pièces de vie et carrelage dans les pièces d'eau,
-             Vendu avec une place de stationnement dans parking fermé et sécurisé en sous-sol (accessible par ascenseur) 

Disponible de su

In [102]:
extracted_new = df["full_description"].str.extract(r'(\d+(?:[.,]\d+)?)\s*m\S*')[0].str.replace(",", ".", regex=False).astype(float)
extracted_new

0       32.00
1       78.75
2      208.00
3         NaN
4         NaN
        ...  
575      6.00
576     73.60
577     85.00
578     78.00
579     59.66
Name: 0, Length: 580, dtype: float64

In [103]:
df["surface"] = df["surface"].fillna(extracted_new)

In [104]:
df[df["titre"]=="Vente Appartement  Sainte-Foy-Les-Lyon"]

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
348,Vente Appartement Sainte-Foy-Les-Lyon,Appartement Sainte Foy Les Lyon 2 pièce(s),Vente Appartement Sainte-Foy-Les-Lyon,Appartement,<NA>,240000,51.6,2,D,"Dans une copropriété calme et sécurisée, un ap...",https://www.etreproprio.com/immobilier-2528632...


In [105]:
df.isna().sum()

titre                 0
description           0
ville                 0
type_bien             0
etage               197
prix                  0
surface               0
pieces                7
DPE                  58
full_description      9
lien                  0
dtype: int64

In [106]:
df.to_csv("annonces_rhone.csv", index=False, encoding="utf-8-sig")

In [107]:
df["full_description"][0]

"Decultieux Immobililer la référence depuis 1962, vous propose en exclusivité à Francheville.\n\nT3 DUPLEX au dernier étage d'un immeuble récent, avec vue dégagée sur les Monts du Lyonnais.\n\nUn appartement très agréable à vivre, calme et lumineux : cuisine toute équipée, un séjour cathédrale de plus de 32m2 ouvrant sur une loggia de 12,5 m2 avec une très belle vue sur les Monts du lyonnais.\n\nL'appartement est climatisé, pour votre confort.\n\nUne suite parentale de 15 m2, avec placards intégrés et une salle de bains.\nUne chambre de 9,2 m2, une buanderie, nombreux rangements et un WC.\n\n\nChauffage individuel par pompe à chaleur pour un diagnostic énergétique performant.\nUne cave complète ce bien.\nLa copropriété est entretenue et sécurisée.\n\nProximité immédiate de tous les commerces à pieds, avec le marché le mardi.\nBus C20 et 14 pour rejoindre Lyon rapidement.\n\nLes informations sur les risques auxquels ce bien est exposé sont disponibles sur le site Géorisques : www.georis

In [108]:
df

,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
0,Vente Appartement 72 m2 Francheville,Appartement 72 m2 à Francheville,Francheville,Appartement,<NA>,269000,72.0,3,D,Decultieux Immobililer la référence depuis 196...,https://www.etreproprio.com/immobilier-2529512...
1,Vente Appartement 78 m2 Villeurbanne,Appartement 78 m2 à Villeurbanne,Villeurbanne,Appartement,3,293000,78.0,4,C,Une exclusivité DECULTIEUX Immobilier\nVILLEUR...,https://www.etreproprio.com/immobilier-2529512...
2,Vente Appartement 208 m2 Feyzin,EXCLUSIVITÉ Feyzin La Bégude Appartement r...,Feyzin,Appartement,0,380000,208.0,5,A,FEYZIN LA BÉGUDE Le confort d’une maison réc...,https://www.etreproprio.com/immobilier-2529551...
3,Vente Appartement 106 m2 Villefranche-Sur-Saone,6732- VILLEFRANCHE S/S Centre ville,Villefranche-Sur-Saone,Appartement,<NA>,239000,106.0,5,D,None,https://www.etreproprio.com/immobilier-2529521...
4,Vente Maison 154 m2 Mions,Maison d’architecte 2022 avec piscine et prest...,Mions,Maison,<NA>,719000,154.0,5,B,"Située au fond d’une impasse, dans un secteur ...",https://www.etreproprio.com/immobilier-2529560...
...,...,...,...,...,...,...,...,...,...,...,...
575,Vente Appartement 64 m2 Lyon-9e,Appartement lumineux avec magnifique vue proch...,Lyon-9e,Appartement,9,285000,64.0,3,C,L'Agence SuitHome vous propose en exclusivité ...,https://www.etreproprio.com/immobilier-2526689...
576,Vente Appartement 75 m2 Lyon-9e,"T3 de 73,6 m2 à Lyon 9 au prix de vente de 182...",Lyon-9e,Appartement,2,182300,75.0,3,C,Découvrez en exclusivité chez Lyon Métropole H...,https://www.etreproprio.com/immobilier-2526819...
577,Vente Appartement 84 m2 Tarare,Appartement 84 m2 à Tarare,Tarare,Appartement,0,120000,84.0,4,D,"A venir découvrir rapidement, à TARARE Centre ...",https://www.etreproprio.com/immobilier-2527082...
578,Vente Appartement 77 m2 Lyon-9e,"Appartement T4 de 78 m2 à Lyon 69009, Rue Féli...",Lyon-9e,Appartement,<NA>,375000,77.0,4,C,"Pour toute question ou demande d'information, ...",https://www.etreproprio.com/immobilier-2526785...


In [111]:
# Filtre les lignes où etage est NaN
df_na = df[df["etage"].isna()]

# Extrait le contexte autour de "étage" dans ces lignes
for desc in df_na["full_description"]:
    if desc:
        matches = re.findall(r'.{0,20}[eéE]tage.{0,20}', str(desc), re.IGNORECASE)
        for m in matches:
            print(repr(m))

"3 DUPLEX au dernier étage d'un immeuble récen"
'tent ce niveau. À l’étage, vous découvrirez u'
'avec lave-mains.À l’étage, place au confort :'
' duplex en dernière étage avec vue 180° sur l'
'paces enfants ou un étage privatisé : tout es'
"TRIANGLE D'OR - En étage élevé d'une résiden"
'Étage : 3ème sur 5 avec a'
'Étage : trois chambres, d'
"loisir de 40 m2 à l'étage, ainsi qu'un espace"
'Étage :'
"te sur le jardinA l'étage, une mezzanine surp"
"partement situé à l'étage 1 d'un immeuble typ"
"lle de bains et à l'étage 4 chambres. 3 autre"
"À l'étage, un magnifique esca"
'vec WC complète cet étage.'
"ne pour accéder à l'étage. 4 chambres planche"
'clusivité - Dernier étage lumineux à Tassin-l'
'm2 situé au dernier étage -- un véritable ato'
'umineux, au dernier étage, idéal pour un coup'
'6,55 m2, en dernier étage avec ascenseur, sta'
"de 90 m2 au dernier étage, bénéficiant d'atou"
'ut situé au dernier étage avec son immense ve'
'À l’étage :\r'
'm2 Carrez, niché en étage élevé, offrant 